# 전처리(Preprocessing)

> **이전 단계(EDA) 요약**
> 1. 타겟 SalePrice는 우편향(skew 1.88) → log 변환으로 정규분포화 필요
> 2. 결측은 두 종류: 구조적("시설 없음" → None) vs 누락(LotFrontage 등 → 추정)
> 3. 가격 핵심 변수는 품질·크기(OverallQual 0.79, GrLivArea 0.71); GrLivArea 이상치 존재

## 전처리 진행 체크리스트

- [x] **이상치 제거** — GrLivArea>4000 & 저가 2채 (Partial 거래)
- [x] **결측 처리** — None(시설없음) / 0(수치 시설없음) / 중앙값(LotFrontage) / 최빈값(Electrical)
- [ ] **타입 교정** — MSSubClass 등 숫자형 가짜 범주 → 범주로
- [ ] **log 변환** — 타겟 SalePrice + 치우친 면적 변수
- [ ] **인코딩** — 범주형 → 숫자 (순서형/명목형 구분)

> TODO: LotFrontage 동네별 중앙값 검토 (현재 전체 중앙값) — 모델 후


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../../data/iowa-housing/train.csv"))
df.shape

In [ ]:
# GrLivArea의 이상치 확인
df[df["GrLivArea"] > 4000][["GrLivArea", "SalePrice", "OverallQual", "SaleCondition"]]

In [ ]:
# "면적 크고 가격 낮은" 비정상 케이스만 제거 (523, 1298)
before = df.shape[0]
df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))]
print(f"{before} → {df.shape[0]}행 ({before - df.shape[0]}개 제거)")


In [ ]:
# 현재 결측 비율 (이상치 제거 후)
missing = df.isnull().mean()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# 결측 있는 컬럼을 타입별로 자동 분류 (확인)
miss = df.columns[df.isnull().any()]
print("범주형:", list(df[miss].select_dtypes("str").columns))
print("수치형:", list(df[miss].select_dtypes("number").columns))

In [ ]:
# LotFrontage 누락 → 중앙값 
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

# Electrical: 1채 누락 → 최빈값
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [ ]:
rest = df.columns[df.isnull().any()]
cat_rest = df[rest].select_dtypes("str").columns
num_rest = df[rest].select_dtypes("number").columns

df[cat_rest] = df[cat_rest].fillna("None")
df[num_rest] = df[num_rest].fillna(0)

print("남은 결측:", df.isnull().sum().sum())

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(font="Malgun Gothic")

sns.histplot(df["LotFrontage"], bins=40)
plt.title("LotFrontage (결측을 중앙값으로 채운 후)")

In [ ]:
orig = pd.read_csv(Path("../../data/iowa-housing/train.csv"))

filled = orig.columns[orig.isnull().any()]
print(f"전체 컬럼: {df.shape[1]}개")
print(f"결측 처리한 컬럼: {len(filled)}개")
print(f"손 안 댄 컬럼: {df.shape[1] - len(filled)}개")
print()
print("처리한 컬럼:", list(filled))

In [ ]:
# 수치형이지만 범주형인 데이터 확인
num = df.select_dtypes("number").columns
df[num].nunique().sort_values()

In [ ]:
# 고유값 24 이하인 수치형 시각화
num = df.select_dtypes("number").columns
low_card = df[num].nunique()
low_card = low_card[low_card <= 24].sort_values().index.tolist()
print(f"{len(low_card)}개:", low_card)

n = len(low_card)
fig, axes = plt.subplots(-(-n // 4), 4, figsize=(20, -(-n // 4) * 3))
for col, ax in zip(low_card, axes.flatten()):
    sns.countplot(x=df[col], ax=ax)
    ax.set_title(f"{col} (고유값 {df[col].nunique()})")
for ax in axes.flatten()[n:]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 0이 많은 변수 확인
num = df.select_dtypes("number").columns
zero_ratio = (df[num] == 0).mean().sort_values(ascending=False)
zero_ratio[zero_ratio > 0]

In [ ]:
# MSSubClass: 주택 유형 코드 → 범주(문자)로 교정
df["MSSubClass"] = df["MSSubClass"].astype(str)
df["MSSubClass"].dtype
df["MSSubClass"].info()